# Financial Needs by Kansas County

This notebook explores healthcare-related financial resource strain across Kansas counties. It joins processed patient information with department records and social-determinant responses, summarizes patient counts by provider, ZIP code, and county, and plots county-level counts for financial resource strain responses.

**Inputs:** `data/private/processed/foot.csv`, `data/private/raw/departments.csv`, and `data/private/processed/sd_clean.csv`.

**Geographic boundaries:** `data/geospatial/kansas_tracts.geojson` and `data/geospatial/kansas_counties.geojson` (used by the optional map block).


In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import folium
import webbrowser

# Find the Bigfoot repository from the current working directory.
current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        folder for folder in (current_dir, *current_dir.parents)
        if (folder / "data").is_dir() and (folder / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the Bigfoot project root. Open Jupyter inside the Bigfoot repository."
    )

RAW_DIR = PROJECT_ROOT / "data" / "private" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "private" / "processed"
GEO_DIR = PROJECT_ROOT / "data" / "geospatial"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
clean_df = pd.read_csv(PROCESSED_DIR / "foot.csv")
clean_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2671593 entries, 0 to 2671592
Data columns (total 30 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Unnamed: 0                   int64  
 1   Date                         str    
 2   EncounterKey                 int64  
 3   PatientDurableKey            int64  
 4   Type_x                       str    
 5   VisitType                    str    
 6   VisitTypeDescription         str    
 7   ProviderDurableKey           int64  
 8   DepartmentKey                int64  
 9   PrimaryDiagnosisKey          int64  
 10  IsEdVisit                    int64  
 11  IsHospitalAdmission          int64  
 12  IsInpatientAdmission         int64  
 13  IsOutpatientFaceToFaceVisit  int64  
 14  CensusBlockGroupFipsCode     int64  
 15  OmbEthnicity                 str    
 16  OmbRace                      str    
 17  SexAssignedAtBirth           str    
 18  VitalStatus                  str    
 19  PatientBirt

In [3]:
map_df=clean_df.copy()
map_df['access_score']=0
map_df['GEOID']=map_df["GEOID"].astype(str)
map_df["county"] = map_df["GEOID"].str[:5]

map_df.head()
map_df.shape

(2671593, 32)

In [4]:
# Load existing Kansas Census GeoJSON files rather than regenerating
# them from machine-specific shapefile paths.
gdf = gpd.read_file(GEO_DIR / "kansas_tracts.geojson")
gdf_kansas = gpd.read_file(GEO_DIR / "kansas_counties.geojson")


In [5]:

"""
# --- Create map ---
m2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles='CartoDB positron')

# --- Census Tracts Layer ---
folium.Choropleth(
    geo_data=str(GEO_DIR / "kansas_tracts.geojson"),
    name="Census Tracts",  
    data=map_df,
    columns=["GEOID", "access_score"],
    key_on="feature.properties.GEOID",
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score"
).add_to(m2)

# --- Counties Layer ---
county_choropleth=folium.Choropleth(
    geo_data=str(GEO_DIR / "kansas_counties.geojson"),
    name="Counties",  
    data=map_df,
    columns=["GEOID", "access_score"],
    key_on="feature.properties.GEOID",
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Medical Access Score",
    show=False  
).add_to(m2)

# --- County popup (click to see name) ---
folium.GeoJsonPopup(
        fields=["NAME"],         
        aliases=["County:"],
        localize=True
    ).add_to(county_choropleth.geojson)


# --- Layer toggle ---
folium.LayerControl().add_to(m2)

# --- Save and open ---
m2.save(str(OUTPUT_DIR / "financial_county_map.html"))
webbrowser.open((OUTPUT_DIR / "financial_county_map.html").resolve().as_uri())"""

'\n# --- Create map ---\nm2 = folium.Map(location=[38.5, -98], zoom_start=7, tiles=\'CartoDB positron\')\n\n# --- Census Tracts Layer ---\nfolium.Choropleth(\n    geo_data=str(GEO_DIR / "kansas_tracts.geojson"),\n    name="Census Tracts",  \n    data=map_df,\n    columns=["GEOID", "access_score"],\n    key_on="feature.properties.GEOID",\n    fill_color=\'YlOrRd\',\n    fill_opacity=0.7,\n    line_opacity=0.2,\n    legend_name="Medical Access Score"\n).add_to(m2)\n\n# --- Counties Layer ---\ncounty_choropleth=folium.Choropleth(\n    geo_data=str(GEO_DIR / "kansas_counties.geojson"),\n    name="Counties",  \n    data=map_df,\n    columns=["GEOID", "access_score"],\n    key_on="feature.properties.GEOID",\n    fill_color=\'YlOrRd\',\n    fill_opacity=0.7,\n    line_opacity=0.2,\n    legend_name="Medical Access Score",\n    show=False  \n).add_to(m2)\n\n# --- County popup (click to see name) ---\nfolium.GeoJsonPopup(\n        fields=["NAME"],         \n        aliases=["County:"],\n        

In [6]:
dept_df = pd.read_csv(RAW_DIR / "departments.csv")


In [7]:
map_df.head()

,Unnamed: 0,Date,EncounterKey,PatientDurableKey,Type_x,VisitType,VisitTypeDescription,ProviderDurableKey,DepartmentKey,PrimaryDiagnosisKey,...,PrimaryDepartment,PrimarySpecialty,GroupCode,DiagnosisValue,GEOID,PopulationValue,CENTLAT,CENTLON,access_score,county
0,0,04/25/22,7848127,86333,Hospital Encounter,*Unspecified,*Unspecified,2982,27,910699,...,*Unspecified,Obstetrics/ Gynecology,ICD-10-CM: O26,O26.893,201770031002,1210.0,39.038987,-95.644925,0,20177
1,1,03/08/22,30915704,385246,Hospital Encounter,*Unspecified,*Unspecified,7973,27,468492,...,SV LINCOLN CENTER,Obstetrics/ Gynecology,ICD-10-CM: O14,O14.04,201770013002,1687.0,39.023433,-95.660754,0,20177
2,2,02/10/22,33972851,6603969,Hospital Encounter,*Unspecified,*Unspecified,6223,27,468646,...,*Unspecified,Obstetrics/ Gynecology,ICD-10-CM: O13,O13.4,200870201014,1629.0,39.194591,-95.510188,0,20087
3,3,01/17/22,45747853,63328,Hospital Encounter,*Unspecified,*Unspecified,2984,27,910699,...,SV LINCOLN CTR URISH,Obstetrics/ Gynecology,ICD-10-CM: O26,O26.893,201770016011,1494.0,39.026401,-95.704505,0,20177
4,4,01/03/22,72680853,5862587,Office Visit,FOLLOW UP,Clinic Visits- In Person,18397,10279,793876,...,ASBURY DERM,Dermatology,ICD-10-CM: D18,D18.01,201770037011,2837.0,38.987558,-95.704307,0,20177


In [8]:
dept_df=dept_df.dropna(subset=['CensusTract'])
dept_df.head()

,DepartmentKey,Address,City,County,DepartmentName,DepartmentSpecialty,DepartmentType,PostalCode,CensusTract
3,1,1500 SW 10th ST,TOPEKA,*Unspecified,INITIAL DEPT,*Unspecified,*Unknown,66604,2.017700e+10
4,2,1979 Milky Way,VERONA,*Unspecified,EHS UM DEPT,*Unspecified,*Unknown,53593,5.502501e+10
5,3,1979 Milky Way,VERONA,*Unspecified,EHS CM DEPT,*Unspecified,*Unknown,53593,5.502501e+10
6,4,1979 Milky Way,VERONA,*Unspecified,EHS CLAIMS DEPT,*Unspecified,*Unknown,53593,5.502501e+10
7,5,1979 Milky Way,VERONA,*Unspecified,EHS PROVIDER DEPT,*Unspecified,*Unknown,53593,5.502501e+10


In [9]:
dept_df['CensusTract']=dept_df['CensusTract'].astype(float).astype('Int64').astype(str)
dept_df.head()

,DepartmentKey,Address,City,County,DepartmentName,DepartmentSpecialty,DepartmentType,PostalCode,CensusTract
3,1,1500 SW 10th ST,TOPEKA,*Unspecified,INITIAL DEPT,*Unspecified,*Unknown,66604,20177003603
4,2,1979 Milky Way,VERONA,*Unspecified,EHS UM DEPT,*Unspecified,*Unknown,53593,55025012700
5,3,1979 Milky Way,VERONA,*Unspecified,EHS CM DEPT,*Unspecified,*Unknown,53593,55025012700
6,4,1979 Milky Way,VERONA,*Unspecified,EHS CLAIMS DEPT,*Unspecified,*Unknown,53593,55025012700
7,5,1979 Milky Way,VERONA,*Unspecified,EHS PROVIDER DEPT,*Unspecified,*Unknown,53593,55025012700


In [10]:
dept_df.dtypes

#pd.merge(map_df,dept_df,left_on='GEOID',right_on='CensusTract')

DepartmentKey          int64
Address                  str
City                     str
County                   str
DepartmentName           str
DepartmentSpecialty      str
DepartmentType           str
PostalCode               str
CensusTract              str
dtype: object

In [11]:
map_df['GEOID'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [12]:
len('201770031002')

12

In [13]:
len('20177003603')

11

In [14]:
map_df.head()
map_df['GEOID'] = map_df['GEOID'].astype(str)

# create tract (first 11 digits)
map_df['tract'] = map_df['GEOID'].str[:11]

# create block (last digit)
map_df['block'] = map_df['GEOID'].str[-1]
map_df.head()

,Unnamed: 0,Date,EncounterKey,PatientDurableKey,Type_x,VisitType,VisitTypeDescription,ProviderDurableKey,DepartmentKey,PrimaryDiagnosisKey,...,GroupCode,DiagnosisValue,GEOID,PopulationValue,CENTLAT,CENTLON,access_score,county,tract,block
0,0,04/25/22,7848127,86333,Hospital Encounter,*Unspecified,*Unspecified,2982,27,910699,...,ICD-10-CM: O26,O26.893,201770031002,1210.0,39.038987,-95.644925,0,20177,20177003100,2
1,1,03/08/22,30915704,385246,Hospital Encounter,*Unspecified,*Unspecified,7973,27,468492,...,ICD-10-CM: O14,O14.04,201770013002,1687.0,39.023433,-95.660754,0,20177,20177001300,2
2,2,02/10/22,33972851,6603969,Hospital Encounter,*Unspecified,*Unspecified,6223,27,468646,...,ICD-10-CM: O13,O13.4,200870201014,1629.0,39.194591,-95.510188,0,20087,20087020101,4
3,3,01/17/22,45747853,63328,Hospital Encounter,*Unspecified,*Unspecified,2984,27,910699,...,ICD-10-CM: O26,O26.893,201770016011,1494.0,39.026401,-95.704505,0,20177,20177001601,1
4,4,01/03/22,72680853,5862587,Office Visit,FOLLOW UP,Clinic Visits- In Person,18397,10279,793876,...,ICD-10-CM: D18,D18.01,201770037011,2837.0,38.987558,-95.704307,0,20177,20177003701,1


In [15]:
merged_df=pd.merge(map_df,dept_df,left_on='tract',right_on='CensusTract')

In [16]:
merged_df.shape

(24916400, 43)

In [17]:
merged_df.head()

,Unnamed: 0,Date,EncounterKey,PatientDurableKey,Type_x,VisitType,VisitTypeDescription,ProviderDurableKey,DepartmentKey_x,PrimaryDiagnosisKey,...,block,DepartmentKey_y,Address,City,County,DepartmentName,DepartmentSpecialty,DepartmentType,PostalCode,CensusTract
0,3,01/17/22,45747853,63328,Hospital Encounter,*Unspecified,*Unspecified,2984,27,910699,...,1,7978,Attn: Phil Salazer MS T-115A,TOPEKA,SHAWNEE,VA HOSPITAL,*Not Applicable,*Not Applicable,66622,20177001601
1,10,03/29/22,76689716,5894283,Office Visit,OFFICE VISIT,Clinic Visit- In Person,18692,10279,728302,...,2,7870,3630 Fairlawn Rd,Topeka,*Unspecified,GENSTLER EYE CENTER,*Not Applicable,*Not Applicable,66610,20177003605
2,10,03/29/22,76689716,5894283,Office Visit,OFFICE VISIT,Clinic Visit- In Person,18692,10279,728302,...,2,7871,3745 SW Wannamaker Rd. Ste A,Topeka,*Unspecified,ELARA CARING - HOSPICE,*Not Applicable,*Not Applicable,66610,20177003605
3,10,03/29/22,76689716,5894283,Office Visit,OFFICE VISIT,Clinic Visit- In Person,18692,10279,728302,...,2,7975,3630 FAIRLAWN RD,TOPEKA,SHAWNEE,TOPEKA SURGERY CENTER,*Not Applicable,*Not Applicable,66610,20177003605
4,10,03/29/22,76689716,5894283,Office Visit,OFFICE VISIT,Clinic Visit- In Person,18692,10279,728302,...,2,7976,3630 FAIRLAWN RD,TOPEKA,SHAWNEE,TOPEKA SURGERY CENTER (AND GENSTLER EYE CENTER),*Not Applicable,*Not Applicable,66610,20177003605


In [18]:
merged_df.columns

Index(['Unnamed: 0', 'Date', 'EncounterKey', 'PatientDurableKey', 'Type_x',
       'VisitType', 'VisitTypeDescription', 'ProviderDurableKey',
       'DepartmentKey_x', 'PrimaryDiagnosisKey', 'IsEdVisit',
       'IsHospitalAdmission', 'IsInpatientAdmission',
       'IsOutpatientFaceToFaceVisit', 'CensusBlockGroupFipsCode',
       'OmbEthnicity', 'OmbRace', 'SexAssignedAtBirth', 'VitalStatus',
       'PatientBirthYearBin', 'ClinicianTitle', 'OfficePostalCode',
       'PrimaryDepartment', 'PrimarySpecialty', 'GroupCode', 'DiagnosisValue',
       'GEOID', 'PopulationValue', 'CENTLAT', 'CENTLON', 'access_score',
       'county', 'tract', 'block', 'DepartmentKey_y', 'Address', 'City',
       'County', 'DepartmentName', 'DepartmentSpecialty', 'DepartmentType',
       'PostalCode', 'CensusTract'],
      dtype='str')

In [19]:
# Count patients per provider
patient_counts = merged_df.groupby('ProviderDurableKey')['PatientDurableKey'].count()

print(patient_counts)

ProviderDurableKey
-1             40
 3            103
 5         108237
 6          14713
 7           1924
            ...  
 491843        84
 492443      3510
 492449        30
 496001      1169
 499028        11
Name: PatientDurableKey, Length: 2024, dtype: int64


In [20]:
# Count unique patients per provider
patient_counts = merged_df.groupby('ProviderDurableKey')['PatientDurableKey'].nunique().reset_index()
patient_counts.columns = ['Provider', 'Patient_Count']
print(patient_counts)

      Provider  Patient_Count
0           -1              4
1            3             14
2            5            221
3            6            457
4            7             91
...        ...            ...
2019    491843             13
2020    492443              9
2021    492449              4
2022    496001              7
2023    499028              2

[2024 rows x 2 columns]


In [21]:
total_patients = merged_df['PatientDurableKey'].nunique()
total_providers = merged_df['ProviderDurableKey'].nunique()

ratio = total_patients / total_providers
print(f"Average Patients per Provider: {ratio:.2f}")


Average Patients per Provider: 20.90


In [22]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(x='Patient_Count', y='Provider', data=patient_counts)
plt.title('Patients Assigned per Provider')
plt.show()


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
merged_df.info()

In [ ]:
# Count unique patients per zipcode
patient_zip = merged_df.groupby('PostalCode')['PatientDurableKey'].nunique().reset_index()
patient_zip.columns = ['PostalCode', 'Patient_Count']
print(patient_zip)

In [ ]:
total_patient_zip = merged_df['PatientDurableKey'].nunique()
total_postal_code = merged_df['PostalCode'].nunique()

ratio = total_patient_zip / total_postal_code
print(f"Average Patients per Postal Code: {ratio:.2f}")

In [ ]:
sns.barplot(x='PostalCode', y='Patient_Count', data=patient_zip)
plt.title('Patients Assigned per Postal Code')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Combine the two columns into one new 'Final_County' column
# This takes 'County' first; if it's missing, it grabs 'county'
merged_df['Final_County'] = merged_df['County'].combine_first(merged_df['county'])

# Optional: Drop the old messy columns to clean up dataframe
merged_df = merged_df.drop(columns=['county', 'County'])

In [ ]:
# Count unique patients per county
patient_county = merged_df.groupby('Final_County')['PatientDurableKey'].nunique().reset_index()
patient_county.columns = ['Final_County', 'Patient_Count']
print(patient_county)

In [ ]:
total_patient_county = merged_df['PatientDurableKey'].nunique()
total_postal_county = merged_df['Final_County'].nunique()

ratio = total_patient_county / total_postal_county
print(f"Average Patients per County: {ratio:.2f}")

In [ ]:
sns.barplot(x='Final_County', y='Patient_Count', data=patient_county)
plt.title('Patients Assigned per County')
plt.xticks(rotation=90)
plt.show()

In [ ]:
financial = pd.read_csv(PROCESSED_DIR / "sd_clean.csv")
financial.info()

In [ ]:
# Filter the dataset to keep only rows where the domain column is 'Financial Needs'
Financial_Filter = financial[financial['Domain'] == 'Financial Resource Strain']
Financial_Filter.info()

In [ ]:
Financial_Filter.drop(Financial_Filter.columns[[0]], axis=1, inplace=True)
Financial_Filter.info()

In [ ]:
# Filter the dataset to keep only rows where the domain column is 'Yes'
Financial_Filter = financial[financial['AnswerText'] == 'Yes']

Financial_Filter.info()

In [ ]:
Financial_Filter.drop(Financial_Filter.columns[[0]], axis=1, inplace=True)
Financial_Filter.info()

In [ ]:
Fin_merged = pd.merge(merged_df, Financial_Filter, on=['PatientDurableKey'], how='inner')
Fin_merged.info()

In [ ]:
"""
# Count unique patients per county for 'Yes'
finance_only = Fin_merged[Fin_merged['AnswerText'] == 'Yes']
result = finance_only.groupby('Final_County')['PatientDurableKey'].nunique()
result"""

In [ ]:
"""result.plot(kind='bar', figsize=(10, 6), color='skyblue', edgecolor='navy')
plt.title('Unique Patients with Financial Needs by County')
plt.xlabel('County')
plt.ylabel('Number of Unique Patients')
plt.xticks(rotation=45) 
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()"""

In [ ]:
print(Fin_merged['Domain'])

In [ ]:
# 1. Filter for the specific domain and the 'yes' response
Fin_merged = Fin_merged[(Fin_merged['Domain'] == 'Financial Resource Strain') & (Fin_merged['AnswerText'] == 'Yes')]

# 2. Group by county and count unique patient IDs
county_counts = Fin_merged.groupby('Final_County')['PatientDurableKey'].nunique().reset_index()

# 3. Rename columns for clarity
county_counts.columns = ['County', 'Unique_Patient_Count']

print(county_counts)

In [ ]:
county_counts.columns

In [ ]:
county_counts.plot(kind='bar', x='County', y='Unique_Patient_Count', figsize=(10, 6), color='skyblue', edgecolor='navy')
plt.title('Unique Patients with Financial Needs by County')
plt.xlabel('County')
plt.ylabel('Number of Unique Patients')
plt.xticks(rotation=45) 
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()